# Reproducing Gao et al. (2018) from dandiset 000007

**Paper.** Gao Z, Davis C, Thomas AM, Economo MN, Abrego AM, Svoboda K, De Zeeuw CI, Li N (2018). A
cortico-cerebellar loop for motor planning. *Nature* 563, 113-116.
[doi:10.1038/s41586-018-0633-x](https://doi.org/10.1038/s41586-018-0633-x). This attempt used the full
text of the author manuscript in PubMed Central (PMC6212318), and the publisher's Supplementary
Information for Supplemental Table 1 (the mice of each experiment).

**Dandiset.** [DANDI:000007](https://dandiarchive.org/dandiset/000007/0.220126.1903), version
0.220126.1903. The paper (2018) cites no DANDI version, as the dandiset was created later. This is the
only published version. The draft holds the same 54 assets (same blobs and sizes) and differs only
in metadata added later (a longer description, keywords, anatomy terms, affiliations), none of which
the analysis uses.

**What the dandiset holds.** ALM recordings in 13 mice, 54 sessions: 6 mice with ChR2 in the
fastigial nucleus and 3 in the dentate nucleus (photo-activation, 34 sessions) and 4 L7-cre x Ai32
mice (CN photo-inhibition, 20 sessions). Spike times, cell type labels and mean waveforms of the
units, and a trial table with the task times, the outcome and a photostimulation code. It does not
hold the CN recordings, the lick times, the lesion and behaviour-only sessions, or the anatomy.

**Outside inputs.** The names of the photostimulation codes come from the data description of the
paper's Zenodo records (doi:10.5281/zenodo.6647629 and 10.5281/zenodo.6653158, found through CRCNS
alm-6, which the paper's reporting summary names); stim-code-link tests them. Method choices the
Methods leave open follow the paper's CN analysis scripts in the MATLAB record. The CN recordings,
also on Zenodo, are analysed in the second notebook, `reproduce_gao_2018_cn.ipynb`.

| Claim | Data | Status | Notebook |
|---|---|---|---|
| stim-code-link | with labels | tested | this |
| methods-alm-unit-counts | dandiset | tested | this |
| fig1g-fn-delay-performance | with labels | tested | this |
| fig1-fn-sample-no-effect | with labels | tested | this |
| fig1g-dn-little-effect | with labels | tested | this |
| fig1h-fn-delay-bias-either-direction | with labels | tested | this |
| ed1f-fn-no-early-licks | with labels | tested | this |
| fig1k-pi-delay-performance | with labels | tested | this |
| fig1d-fn-lesion | dandiset | blocked, data missing | |
| fig1e-lesion-lick-timing | dandiset | blocked, data missing | |
| fig1h-response-contra-bias | dandiset | blocked, data missing | |
| fig1i-licking-unaffected | dandiset | blocked, data missing | |
| fig1k-response-ipsi-bias | dandiset | blocked, data missing | |
| ed2-pi-silences-fastigial | dandiset | blocked, data missing | |
| fig2-alm-selective-fraction | dandiset | tested | this |
| fig2-alm-preparatory-fraction | dandiset | tested | this |
| fig2-alm-perimovement-fraction | dandiset | tested | this |
| fig2c-alm-selectivity-time-course | dandiset | tested | this |
| fig2-cn-unit-counts | elsewhere (Zenodo) | tested | CN |
| fig2-cn-selective-fraction | elsewhere (Zenodo) | tested | CN |
| fig2-cn-preparatory-fraction | elsewhere (Zenodo) | tested | CN |
| fig2-cn-perimovement-fraction | elsewhere (Zenodo) | tested | CN |
| ed4e-cn-predict-movement | elsewhere (Zenodo) | tested | CN |
| ed4g-fn-contra-late-delay | elsewhere (Zenodo) | tested | CN |
| ed4c-lick-modulation | elsewhere (Zenodo) | blocked, too large | CN |
| fig3b-cn-modulated-fraction | elsewhere (Zenodo) | tested | CN |
| fig3b-cn-net-increase | elsewhere (Zenodo) | tested | CN |
| fig3-cn-latency | elsewhere (Zenodo) | tested | CN |
| fig3c-cn-selectivity-abolished | elsewhere (Zenodo) | tested | CN |
| ed6-cn-either-hemisphere | elsewhere (Zenodo) | tested | CN |
| fig3e-alm-modulated-fraction | with labels | tested | this |
| fig3e-alm-mean-rate-unchanged | with labels | tested | this |
| fig3e-alm-up-down | with labels | tested | this |
| fig3-alm-latency-pi | with labels | tested | this |
| fig3f-alm-selectivity-abolished | with labels | tested | this |
| ed9a-fn-latency | with labels | tested | this |
| ed9a-dn-latency | with labels | tested | this |
| fig4f-fn-persistent | with labels | tested | this |
| fig4g-dn-recovery | with labels | tested | this |
| fig4i-fn-cd-collapse | with labels | tested | this |
| fig4i-dn-cd-minimal | with labels | tested | this |
| ed10d-fn-no-prediction | with labels | tested | this |
| ed10c-alm-pi-endpoints | dandiset | blocked, data missing | |
| ed10efg-alm-needed-after-fn | dandiset | blocked, data missing | |
| fig4bc-fastigial-thalamus | dandiset | blocked, data missing | |

Seeds are fixed per section. The notebook streams about 200 MB and takes about six minutes.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

import gao2018

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sessions = gao2018.load_all()
print(f"loaded {len(sessions)} files from dandiset {gao2018.DANDISET_ID} version {gao2018.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
inv = pd.DataFrame([{
    "group": s["group"], "subject": s["subject"], "session": s["session"], "genotype": s["genotype"],
    "site": f'{s["site"]} {s["site_hemisphere"]}',
    "virus": ";".join(f'{v["brain_location"]} {v["hemisphere"]}' for v in s["virus"]),
    "probe": s["probe"], "rec_ml": s["rec_ml"], "trials": len(s["trials"]), "units": len(s["spikes"]),
    "pyramidal": int((s["cell_type"] == "pyramidal").sum()), "FS": int((s["cell_type"] == "FS").sum()),
    "other_label": int((~np.isin(s["cell_type"], ["pyramidal", "FS"])).sum())} for s in sessions])
print("Groups: FN = ChR2 virus in the fastigial nucleus, DN = in the dentate nucleus, PI = L7-cre x Ai32 (CN photo-inhibition)")
print(inv.groupby(["group", "subject"]).agg(sessions=("session", "size"), trials=("trials", "sum"), units=("units", "sum"),
      pyramidal=("pyramidal", "sum"), FS=("FS", "sum"), other=("other_label", "sum")).to_string())
print("\nTotals by group:")
print(inv.groupby("group")[["trials", "units", "pyramidal", "FS", "other_label"]].sum()
      .assign(sessions=inv.groupby("group").size()).to_string())
print("\nSubjects:", inv.subject.nunique(), "; sessions:", len(inv), "; cell type labels:",
      sorted(set(np.concatenate([s["cell_type"] for s in sessions]))))
print("\nPer subject: genotype, stimulation site, virus, probe, recording ML (mm from midline):")
print(inv.drop_duplicates("subject")[["group", "subject", "genotype", "site", "virus", "probe", "rec_ml"]].to_string(index=False))
alltr = pd.concat([s["trials"].assign(group=s["group"], subject=s["subject"]) for s in sessions], ignore_index=True)
print("\nTrials by group and photo_stim_id:")
print(pd.crosstab(alltr.group, alltr.stim).to_string())
print("\nTrials by response:", alltr.response.value_counts().to_dict())
print("Early-lick trials:", int(alltr.lick_early.sum()), "of", len(alltr))
print("Trial table columns:", list(sessions[0]["trials_columns"]) if "trials_columns" in sessions[0] else
      list(sessions[0]["trial_descriptions"]))
print("Units table columns:", sessions[0]["units_colnames"])

The archive lists 13 subjects and 54 files, and the files hold 13 mice: 6 with ChR2 virus in the
fastigial nucleus (FN), 3 in the dentate nucleus (DN) and 4 L7-cre x Ai32 mice for CN
photo-inhibition (PI). This matches the ALM recordings of Supplemental Table 1 (6, 3 and 4 mice).
The FN and DN groups hold 34 sessions and the PI group 20, as the Zenodo data description says;
the Methods give 19 sessions for the PI group. The dandiset holds no CN recordings, no lick
times, no lesion or behaviour-only sessions, and no anatomy. The `photo_stim_id` codes (0 to 6)
have no description in the files or the dandiset metadata. Codes 1 and 2 occur only in FN mice,
3 and 4 only in DN mice, and 5 and 6 only in PI mice. The trial table has no column marking the
periods when the mouse was not performing (`GoodTrials` in the paper's scripts); the many
`NoLick` trials at the ends of some sessions are such periods, and they drop out of performance
and of the correct-trial analyses here.

In [ ]:
# Units and spike widths. The Methods classify units by trough-to-peak width: below 0.35 ms fast
# spiking, above 0.45 ms putative pyramidal. The files give a cell type label, a sampling rate for
# the waveform and a 30-sample mean waveform.
rows = []
for s in sessions:
    for ct, wf, hz in zip(s["cell_type"], s["waveform"], s["unit_rate_hz"]):
        rows.append({"group": s["group"], "probe": s["probe"][:12], "label": ct, "rate_hz": int(hz),
                     "width_ms": round(gao2018.spike_width_ms(wf, hz), 3)})
units = pd.DataFrame(rows)
print("Waveform sampling rate stated, by probe:", units.groupby("probe").rate_hz.unique().to_dict())
print("Methods: 32-channel NeuroNexus recordings were sampled at 25,000 Hz, 64-channel at 20,000 Hz.")
print("\nWidth (ms) by label, at the stated rate:")
print(units.groupby(["probe", "label"]).width_ms.describe()[["count", "min", "25%", "50%", "75%", "max"]].to_string())
units["width_class"] = pd.cut(units.width_ms, [0, gao2018.NARROW_MS - 1e-9, gao2018.WIDE_MS + 1e-9, 9],
                              labels=["narrow (<0.35)", "intermediate", "wide (>0.45)"])
print("\nLabel against width class, at the stated rate:")
print(pd.crosstab([units.probe, units.label], units.width_class).to_string())
scaled = units.width_ms * np.where(units.probe.str.startswith("NN"), 20000 / 25000, 1.0)
units["width_class_25k"] = pd.cut(scaled, [0, gao2018.NARROW_MS - 1e-9, gao2018.WIDE_MS + 1e-9, 9],
                                  labels=["narrow (<0.35)", "intermediate", "wide (>0.45)"])
print("\nThe same if the NeuroNexus waveforms were sampled at 25 kHz:")
print(pd.crosstab([units.probe, units.label], units.width_class_25k).to_string())
e = sessions[0]
print("\nUnit observation intervals, first unit of each of 3 files:", [s["obs"][0].round(1).tolist() for s in sessions[:3]])

The files' labels follow the width rule closely but not exactly: a few units labelled pyramidal
have widths of 0.30 ms or less at the stated rate. The stated waveform rate is 20 kHz for every
unit, including the NeuroNexus recordings that the Methods say were sampled at 25 kHz; with 25
kHz the labels agree with the width classes in the same way (printed above), so the rate cannot
be settled from the widths. The analyses below use the files' labels; the width rule is a
variant in methods-alm-unit-counts. The electrode table gives no positions (x, y and z are NaN)
and no impedances (-1), so unit depth is not available.

In [ ]:
# Trial times. The table's descriptions say pole_in_time, pole_out_time and cue_time are relative
# to the trial start; the values are compared with start_time here.
d = sessions[0]["trial_descriptions"]
for c in ("start_time", "pole_in_time", "pole_out_time", "cue_time"):
    print(f"{c}: {d[c]!r}")
lag = pd.DataFrame({
    "pole_in - start": alltr.pole_in_time - alltr.start_time,
    "sample (pole_out - pole_in)": alltr.pole_out_time - alltr.pole_in_time,
    "delay (cue - pole_out)": alltr.cue_time - alltr.pole_out_time,
    "trial (stop - start)": alltr.stop_time - alltr.start_time})
print("\n", lag.describe().round(3).to_string())
print("\npole_in_time range over all trials:", alltr.pole_in_time.min().round(1), "to", alltr.pole_in_time.max().round(1), "s")
in_trials = []
for s in sessions:
    sp = np.concatenate(s["spikes"])
    n_in = gao2018.counts(np.sort(sp), s["trials"].start_time, s["trials"].stop_time).sum()
    in_trials.append(n_in / len(sp))
print("Fraction of spikes inside the 5 s trial windows, per file: median", np.round(np.median(in_trials), 3),
      "range", np.round(min(in_trials), 3), "to", np.round(max(in_trials), 3))
print("Spike times sorted within every unit:", all(np.all(np.diff(sp) >= 0) for s in sessions for sp in s["spikes"]))

The trial times are not relative to the trial start: pole in follows `start_time` by 0.57 s in
every trial, and all the times run over the session (up to thousands of seconds), on the clock of
the spike times. The descriptions are wrong; the analyses use the times as session times. The
sample and delay epochs are 1.3 s in every trial, as the Methods state. Each trial row covers a
fixed 5 s window from 0.57 s before pole in, so `stop_time` is a window end, not the end of the
trial. Spike times are sorted.

In [ ]:
# Supplemental Table 1 lists the mice and hemispheres of each experiment. Its ALM recording rows,
# set against the files (ML coordinates are negative on the left; the 2015 mice have numeric ids,
# and only anm00314757, the one with a left fastigial site and right ALM, can be matched to the
# table's AN107, and anm00314756, the one dentate mouse with a numeric id, to AN106).
table = pd.DataFrame([
    ("BAYLORCD8", "left", "right"), ("BAYLORCD12", "left", "right"), ("BAYLORCD13", "left", "right"),
    ("BAYLORCD14", "right", "left"), ("BAYLORCD6", "left", "right"), ("BAYLORNL15", "left", "right"),
    ("anm00314757", "right", "left"), ("BAYLORNL12", "left", "right"), ("BAYLORNL14", "left", "right"),
    ("anm00314756", "left", "right")], columns=["subject", "table_ALM", "table_CN"])
files = inv.drop_duplicates("subject").set_index("subject")
table["file_ALM"] = table.subject.map(lambda m: "left" if files.loc[m, "rec_ml"] < 0 else "right")
table["file_CN"] = table.subject.map(lambda m: files.loc[m, "site"].split()[1])
table["agrees"] = (table.table_ALM == table.file_ALM) & (table.table_CN == table.file_CN)
print(table.to_string(index=False))
print("\nThe Methods: 'ALM recordings were always contralateral to the side of CN manipulations'.")
print("Files with ALM and CN site on the same side:",
      [m for m, r in files.iterrows() if (r.rec_ml < 0) == (r.site.split()[1] == "left")])
pi = files[files.group == "PI"]
print("\nPI mice, stimulation site coordinates as given in the files:")
for s in sessions:
    if s["group"] == "PI" and s["session"] == min(x["session"] for x in sessions if x["subject"] == s["subject"]):
        print(" ", s["subject"], s["opto"])

For the ALM recordings during CN photo-inhibition the files disagree with Supplemental Table 1 in
three of four mice: BAYLORCD12 and BAYLORCD13 are given a left fastigial site where the table
has the right CN, and BAYLORCD14 is given left ALM where the table has right ALM. In those three
files ALM and the stimulation site are on the same side, against the Methods' statement that ALM
recordings were always contralateral to the CN manipulation. The PI files also describe the
photostimulation site as the fastigial nucleus at 2.4 mm depth, while the Methods photostimulate
the cerebellar cortex through a cranial window at the brain surface. For the FN and DN mice the
files agree with the table where the mice can be matched. No analysis below uses the side of
photo-inhibition; fig1h-fn-delay-bias-either-direction uses the FN side, which agrees.

In [ ]:
# Recordings deposited twice: identical or near-identical spike trains across all units, and
# repeated trial tables.
import hashlib
from collections import defaultdict
digests = defaultdict(list)
for s in sessions:
    for i, sp in enumerate(s["spikes"]):
        digests[hashlib.md5(np.round(sp, 5).tobytes()).hexdigest()].append((s["subject"], s["session"], i))
print("Groups of identical spike trains:", [v for v in digests.values() if len(v) > 1])
tables = defaultdict(list)
for s in sessions:
    tables[hashlib.md5(np.round(s["trials"][["pole_in_time", "cue_time"]].values, 3).tobytes()).hexdigest()].append(s["session"])
print("Repeated trial tables:", [v for v in tables.values() if len(v) > 1])
near = []
for s in sessions:
    sp = s["spikes"]
    for i in range(len(sp)):
        for j in range(i + 1, len(sp)):
            a, b = (sp[i], sp[j]) if len(sp[i]) <= len(sp[j]) else (sp[j], sp[i])
            if len(a) < 100:
                continue
            k = np.clip(np.searchsorted(b, a), 1, len(b) - 1)
            frac = (np.minimum(np.abs(b[k] - a), np.abs(b[k - 1] - a)) < 0.0005).mean()
            if frac > 0.5:
                near.append((s["subject"], s["session"], i, j, len(sp[i]), len(sp[j]), round(frac, 2)))
print("Unit pairs in one session sharing more than half of the smaller unit's spikes within 0.5 ms:")
print(pd.DataFrame(near, columns=["subject", "session", "unit_a", "unit_b", "spikes_a", "spikes_b", "shared"]).to_string(index=False))

No recording is deposited twice: no two files share a trial table and no two units share a spike
train. Within two sessions a pair of units shares most spikes (83% and 96% of the smaller unit's
spikes within 0.5 ms), so each pair is likely one neuron sorted twice or a unit and part of it.
These four units are kept, as the paper's counts give no way to tell which were used; they are 4
of 1923 units. Units are not matched across sessions; each session's units are treated as
distinct neurons, as the paper's counts do.

In [ ]:
# Clock check 1: the trial table and the spikes share a clock. Decoding the instructed trial type
# from population spike counts (200 ms windows) in each session, with 5-fold cross-validated
# logistic regression on control trials, should be at chance before pole in and rise after it.
centres = np.arange(-0.4, 3.5, 0.2)
rng_clock = np.random.default_rng(1)
acc = []
for s in sessions:
    tr = s["trials"]
    ok = (tr.stim == 0) & (tr.lick_early == 0) & tr.outcome.isin(["hit", "error"])
    t0, y = tr.pole_in_time.values[ok], (tr.trial_type.values[ok] == "R").astype(int)
    if min(y.sum(), (1 - y).sum()) < 10:
        continue
    X = np.stack([gao2018.binned_counts(sp, t0, centres + 0.1, 0.2) for sp in s["spikes"]], axis=2)
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    row = {"session": f'{s["subject"]} {s["session"]}', "group": s["group"]}
    for k, c in enumerate(centres):
        row[round(c, 1)] = cross_val_score(LogisticRegression(C=0.1, max_iter=2000), X[:, k, :], y, cv=cv,
                                           scoring="balanced_accuracy").mean()
    acc.append(row)
acc = pd.DataFrame(acc)
cols = [c for c in acc.columns if isinstance(c, float)]
print("Balanced accuracy, median over sessions, by window start relative to pole in (s):")
print(acc[cols].median().round(2).to_string())
pre = acc[[c for c in cols if c < 0]].mean(axis=1)
late_delay = acc[[c for c in cols if 2.0 <= c < 2.6]].mean(axis=1)
print(f"\nSessions decoded: {len(acc)} of {len(sessions)}")
print("Mean accuracy before pole in: median", pre.median().round(3), "; in the last 600 ms of the delay: median",
      late_delay.median().round(3))
print("Sessions whose late-delay accuracy exceeds the pre-trial accuracy by 0.1 or more:",
      int((late_delay - pre >= 0.1).sum()), "of", len(acc))
print("Sessions where it does not:", acc.session[(late_delay - pre < 0.1)].tolist())

Before pole in the trial type cannot be decoded (median balanced accuracy near 0.5), and it rises
during the sample and delay epochs, so the trial table and the spikes share a clock at the scale
of the epochs. The sessions listed above, where the delay is not decoded better than the
pre-trial period, have few units or trials; for them the shared clock is not shown by this
check. The photostimulation responses below (stim-code-link) start within milliseconds of the
stimulus onset taken from the trial table, which shows the alignment at a finer scale for every
group of mice.

In [ ]:
# The R and L of `response`. In the paper's scripts "yes" trials are HitR or ErrR, so R is the
# instructed side. NoLickR and NoLickL also carry a side, which can only be the instructed one. A
# check from the data: ALM late-delay activity follows the upcoming lick on error trials (Extended
# Data Fig. 4e, and earlier work of the lab), so if R is the instructed side, ErrR trials (licked
# left) should resemble HitL trials.
rows = []
for s in sessions:
    tr = s["trials"]
    ok = (tr.stim == 0) & (tr.lick_early == 0)
    a, b = tr.cue_time.values - 0.6, tr.cue_time.values
    X = np.stack([gao2018.counts(sp, a, b) / 0.6 for sp in s["spikes"]], axis=1)
    hr, hl = ok & (tr.response == "HitR"), ok & (tr.response == "HitL")
    er, el = ok & (tr.response == "ErrR"), ok & (tr.response == "ErrL")
    if min(er.sum(), el.sum()) < 3:
        continue
    w = X[hr].mean(0) - X[hl].mean(0)
    mid = (X[hr].mean(0) + X[hl].mean(0)) @ w / 2
    rows.append({"session": s["session"], "ErrR_like_HitR": float(((X[er] @ w) > mid).mean()),
                 "ErrL_like_HitR": float(((X[el] @ w) > mid).mean()), "n_ErrR": int(er.sum()), "n_ErrL": int(el.sum())})
err = pd.DataFrame(rows)
print("Fraction of error trials whose last-600-ms delay activity falls on the HitR side of the HitR/HitL midpoint, per session:")
print(err[["ErrR_like_HitR", "ErrL_like_HitR"]].describe().round(3).to_string())
print("Sessions where ErrR trials fall on the HitR side less often than ErrL trials:",
      int((err.ErrR_like_HitR < err.ErrL_like_HitR).sum()), "of", len(err))

Of the trials, 1244 have a sample or delay epoch that differs from 1.3 s by more than 50 ms; all
but one of them are early-lick trials, in which the task restarts the epoch, and they are left
out with the early-lick trials. 

In [ ]:
smp = alltr.pole_out_time - alltr.pole_in_time
dly = alltr.cue_time - alltr.pole_out_time
odd = (abs(smp - 1.3) > 0.05) | (abs(dly - 1.3) > 0.05)
print("Trials with a sample or delay epoch more than 50 ms from 1.3 s:", int(odd.sum()),
      "; of them early-lick trials:", int(alltr.lick_early[odd].sum()))
print("Among trials without early licks, sample epoch", smp[alltr.lick_early == 0].round(3).agg(["min", "max"]).tolist(),
      "s, delay epoch", dly[alltr.lick_early == 0].round(3).agg(["min", "max"]).tolist(), "s")

In [ ]:
# Precision of the trial event times: the fraction of values that fall on a 10 ms grid, per mouse.
prec = []
for s in sessions:
    t = s["trials"]
    row = {"group": s["group"], "subject": s["subject"]}
    for c in ("pole_in_time", "pole_out_time", "cue_time"):
        v = t[c].values
        row[c] = float((np.abs(v * 100 - np.round(v * 100)) < 1e-6).mean())
    prec.append(row)
prec = pd.DataFrame(prec).groupby(["group", "subject"]).mean().round(2)
print("Fraction of event times on a 10 ms grid:")
print(prec.to_string())

In the files of the 2017 and 2018 mice (BAYLOR ids, 40 of 54 sessions, all four PI mice) the pole
and cue times lie on a 10 ms grid, while in the 2015 files (anm ids) they are given to a
microsecond. Times of the photostimulus are known only through these event times, so in the
BAYLOR files an onset latency of a few milliseconds cannot be measured to better than about 10
ms. The latency claims show the 2015 files apart as a variant.

## stim-code-link

> The dandiset's photostimulation codes 1 to 6 can be named from the paper's Zenodo data
> description (1 and 2 fastigial photo-activation in the sample and delay epochs, 3 and 4 dentate
> photo-activation in the sample and delay, 5 and 6 CN photo-inhibition in the delay for 0.5 and
> 1.3 s), and the ALM responses agree with those names.

The names are in `gao2018.STIM_CODES`, from the data description of the Zenodo records. The ALM
response to each code is the mean over all units of the group of the absolute difference in
firing rate between that code's trials and control trials (no early licks), in 10 ms bins
aligned to pole in.

In [ ]:
BIN = 0.01
link_edges = np.arange(-0.5, 3.6 + BIN / 2, BIN)
link_t = link_edges[:-1] + BIN / 2
group_codes = {"FN": (1, 2), "DN": (3, 4), "PI": (5, 6)}
change = {}
for grp, codes in group_codes.items():
    per_unit = {c: [] for c in codes}
    for s in sessions:
        if s["group"] != grp:
            continue
        tr = s["trials"][s["trials"].lick_early == 0]
        for sp in s["spikes"]:
            rate = {}
            for c in (0,) + codes:
                t0 = tr.pole_in_time.values[tr.stim.values == c]
                if len(t0) == 0:
                    continue
                rate[c] = np.mean([np.histogram(sp[(sp > t + link_edges[0]) & (sp < t + link_edges[-1])] - t, link_edges)[0]
                                   for t in t0], axis=0) / BIN
            for c in codes:
                if (tr.stim.values == c).sum() > 0:
                    per_unit[c].append(np.abs(rate[c] - rate[0]))
    for c in codes:
        change[c] = np.mean(per_unit[c], axis=0)
        change[(c, "n_units")] = len(per_unit[c])

onset_expected = {1: 0.0, 2: 1.3, 3: 0.0, 4: 1.3, 5: 1.3, 6: 1.3}
rows = []
for c, ref in onset_expected.items():
    x = change[c]
    base = x[link_t < 0]
    thr = base.mean() + 4 * base.std()
    above = link_t[(x > thr) & (link_t >= 0)]
    first = above[0] if len(above) else np.nan
    rows.append({"code": c, "name": " ".join(str(v) for v in gao2018.STIM_CODES[c][1:3]), "units": change[(c, "n_units")],
                 "stated_onset_s": ref, "first_bin_above_s": round(first, 3),
                 "first_minus_onset_ms": round((first - ref) * 1000, 1),
                 "bins_above_before_onset": int(((x > thr) & (link_t >= 0) & (link_t < ref)).sum()),
                 "change_0.7_1.2s_after_onset": round(x[(link_t >= ref + 0.7) & (link_t < ref + 1.2)].mean() - base.mean(), 3)})
link = pd.DataFrame(rows)
print("Time from pole in of the first 10 ms bin (centre) whose change exceeds the pre-trial mean by 4 SD:")
print(link.to_string(index=False))
codes_by_group = alltr.groupby("group").stim.unique().map(sorted)
print("\nCodes present in each group:", codes_by_group.to_dict())
c1 = all(set(codes_by_group[g]) == {0, *group_codes[g]} for g in group_codes)
c2 = bool(((link.first_minus_onset_ms >= 0) & (link.first_minus_onset_ms <= 50) & (link.bins_above_before_onset == 0)).all())
c3 = bool(link.set_index("code").loc[5, "change_0.7_1.2s_after_onset"] < link.set_index("code").loc[6, "change_0.7_1.2s_after_onset"])
print(f"\n(1) codes only in their groups: {c1}; (2) onsets within 50 ms after the stated onset and none before: {c2}; "
      f"(3) code 5 ends before code 6: {c3}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, (grp, codes) in zip(axes, group_codes.items()):
    for c in codes:
        ax.plot(link_t, change[c], label=f"code {c}")
    for x in (0, 1.3, 2.6):
        ax.axvline(x, color="k", lw=0.5)
    ax.set_title(grp); ax.set_xlabel("time from pole in (s)"); ax.legend()
axes[0].set_ylabel("mean |rate change| (spikes/s)")
plt.tight_layout(); plt.show()
print("Plotted range:", round(min(change[c].min() for c in onset_expected), 2), "to",
      round(max(change[c].max() for c in onset_expected), 2), "spikes/s; all points inside the axes")

**Result: reproduced.** Codes 1 and 2 occur only in the six fastigial mice, 3 and 4 only in the
three dentate mice and 5 and 6 only in the four L7-cre mice. For every code the ALM change first
exceeds the pre-trial level by 4 SD 5 to 35 ms after the onset the name implies (pole in for codes
1 and 3, pole out for 2, 4, 5 and 6), with no bin above it between pole in and that onset. The
change 0.7 to 1.2 s after onset is 0.95 spikes/s after code 5 and 1.64 after code 6, so code 5
ends earlier, as a 0.5 s photostimulus against a 1.3 s one would. The names from the Zenodo data
description are used below.

## methods-alm-unit-counts

> The ALM recordings comprise 912 putative pyramidal neurons in the 9 mice tested with CN
> photo-activation (446 with fastigial and 466 with dentate photo-activation), and 490 single
> units (454 putative pyramidal) in 4 mice across 19 sessions during CN photo-inhibition.

In [ ]:
units["subject"] = np.concatenate([[s["subject"]] * len(s["spikes"]) for s in sessions])
wide = units.width_ms > gao2018.WIDE_MS
counts_tbl = pd.DataFrame({
    "expected": [912, 446, 466, 490, 454, 19],
    "(a) labels": [int((units.label == "pyramidal")[units.group.isin(["FN", "DN"])].sum()),
                   int((units.label == "pyramidal")[units.group == "FN"].sum()),
                   int((units.label == "pyramidal")[units.group == "DN"].sum()),
                   int((units.group == "PI").sum()), int((units.label == "pyramidal")[units.group == "PI"].sum()),
                   int((inv.group == "PI").sum())],
    "(b) width rule": [int(wide[units.group.isin(["FN", "DN"])].sum()), int(wide[units.group == "FN"].sum()),
                       int(wide[units.group == "DN"].sum()), int((units.group == "PI").sum()),
                       int(wide[units.group == "PI"].sum()), int((inv.group == "PI").sum())]},
    index=["pyramidal, photo-activation mice", "pyramidal, fastigial mice", "pyramidal, dentate mice",
           "units, photo-inhibition mice", "pyramidal, photo-inhibition mice", "sessions, photo-inhibition mice"])
counts_tbl["tolerance"] = (counts_tbl.expected * 0.1).round(1)
for r in ("(a) labels", "(b) width rule"):
    counts_tbl[f"within {r[:3]}"] = (counts_tbl[r] - counts_tbl.expected).abs() <= counts_tbl.tolerance
print(counts_tbl.to_string())
print("\nPyramidal units (labels) per mouse:")
print(units[units.label == "pyramidal"].groupby(["group", "subject"]).size().to_string())
print("\nPaper: 1309 ALM units in 10 mice during behaviour (1194 pyramidal, 82 FS, 33 intermediate);"
      " dandiset, 9 photo-activation mice:", int(units.group.isin(["FN", "DN"]).sum()), "units,",
      int(((units.label == "FS") & units.group.isin(["FN", "DN"])).sum()), "FS")

**Result: not reproduced.** Only the session count of the photo-inhibition mice (20 against 19)
is within tolerance, under both readings. The dandiset holds more units than the paper reports:
1112 pyramidal neurons in the 9 photo-activation mice against 912 (567 against 446 in the
fastigial mice, 545 against 466 in the dentate mice), and 737 units (706 pyramidal) in the
photo-inhibition mice against 490 (454). The width rule gives similar counts (1082, 557, 525,
676). With the 9 mice the dandiset already has 1186 units, against the paper's 1309 for 10 mice.
The paper's unit selection is not described and the dandiset does not mark which units the paper
used.

## Behaviour during the recordings

The behavioural claims are tested on the trials of the recording sessions, the only behaviour in
the dandiset. Performance is the fraction correct without early-lick and no-lick trials. A
contralateral trial is one whose instructed lick is toward the side opposite the stimulated
nucleus. `perf_data` gathers, per mouse and session, each trial's outcome and condition for the
hierarchical bootstrap of the Methods.

In [ ]:
def perf_table(group):
    rows = []
    for s in sessions:
        if s["group"] != group:
            continue
        t = gao2018.performance(s["trials"])
        contra_side = "L" if s["site_hemisphere"] == "right" else "R"
        rows.append(t.assign(subject=s["subject"], session=s["session"],
                             side=np.where(t.trial_type == contra_side, "contra", "ipsi")))
    return pd.concat(rows, ignore_index=True)


def perf_data(tbl, condition):
    """{mouse: [(correct, condition index), ...]} for trials whose condition is not None."""
    data = {}
    for (m, sess), g in tbl.groupby(["subject", "session"]):
        k = condition(g)
        ok = k >= 0
        if ok.sum():
            data.setdefault(m, []).append((g.correct.values[ok], k[ok].astype(int)))
    return data


def perf_change(boot, stim=1, ctrl=0):
    return boot[:, stim, 0] / boot[:, stim, 1] - boot[:, ctrl, 0] / boot[:, ctrl, 1]


perf_fn, perf_dn, perf_pi = perf_table("FN"), perf_table("DN"), perf_table("PI")
for name, t in (("FN", perf_fn), ("DN", perf_dn), ("PI", perf_pi)):
    print(name, "performance by mouse and code (fraction correct, trials):")
    print(t.groupby(["subject", "stim"]).correct.agg(["mean", "size"]).unstack("stim").round(3).to_string(), "\n")

## fig1g-fn-delay-performance

> Fastigial photo-activation during the early delay epoch reduced behavioural performance
> (incorrect choices).

In [ ]:
rng = np.random.default_rng(101)
obs = perf_fn[perf_fn.stim == 2].correct.mean() - perf_fn[perf_fn.stim == 0].correct.mean()
boot = gao2018.hierarchical_bootstrap(perf_data(perf_fn, lambda g: np.select([g.stim == 0, g.stim == 2], [0, 1], -1)),
                                      2, 10000, rng)
d = perf_change(boot)
print(f"FN mice ({perf_fn.subject.nunique()}), code 2 against control: control {perf_fn[perf_fn.stim == 0].correct.mean():.3f}"
      f" ({(perf_fn.stim == 0).sum()} trials), photostim {perf_fn[perf_fn.stim == 2].correct.mean():.3f} ({(perf_fn.stim == 2).sum()} trials)")
print(f"change {obs:+.3f}; bootstrap 95% interval {np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f}; "
      f"one-tailed p = {gao2018.one_tailed_p(obs, d):.4f}")
per_mouse = perf_fn[perf_fn.stim.isin([0, 2])].groupby(["subject", "stim"]).correct.mean().unstack()
per_mouse["change"] = per_mouse[2] - per_mouse[0]
print("\nPer mouse:"); print(per_mouse.round(3).to_string())
print("Mice with lower performance on code 2 trials:", int((per_mouse.change < 0).sum()), "of", len(per_mouse))

**Result: reproduced.** In the six fastigial mice, delay-epoch photo-activation lowers
performance from 0.873 to 0.630 (change -0.243, 95% interval -0.388 to -0.101, one-tailed
bootstrap p = 0.0001), and performance is lower on code 2 trials in all six mice.

## fig1-fn-sample-no-effect

> Fastigial photo-activation during the early sample epoch had no effect on behavioural choice
> (p > 0.05 for either trial type, bootstrap).

In [ ]:
rng = np.random.default_rng(102)
rows = []
for side in ("contra", "ipsi"):
    sub = perf_fn[perf_fn.side == side]
    obs = sub[sub.stim == 1].correct.mean() - sub[sub.stim == 0].correct.mean()
    boot = gao2018.hierarchical_bootstrap(perf_data(sub, lambda g: np.select([g.stim == 0, g.stim == 1], [0, 1], -1)),
                                          2, 10000, rng)
    d = perf_change(boot)
    rows.append({"trial type": side, "control": sub[sub.stim == 0].correct.mean(), "code 1": sub[sub.stim == 1].correct.mean(),
                 "change": obs, "ci_low": np.percentile(d, 2.5), "ci_high": np.percentile(d, 97.5),
                 "one_tailed_p": gao2018.one_tailed_p(obs, d), "n_code1": int((sub.stim == 1).sum())})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("By lick direction, the same test:")
for side in ("R", "L"):
    sub = perf_fn[perf_fn.trial_type == side]
    print(f"  lick {side}: control {sub[sub.stim == 0].correct.mean():.3f}, code 1 {sub[sub.stim == 1].correct.mean():.3f}")

**Result: reproduced.** Sample-epoch fastigial photo-activation changes performance by -0.034
on contralateral trials (one-tailed p = 0.365) and +0.024 on ipsilateral trials (p = 0.253), so
neither trial type shows a decrease with p < 0.05.

## fig1g-dn-little-effect

> Dentate photo-activation produced little effect on behavioural choice (p = 0.06, bootstrap).

In [ ]:
rng = np.random.default_rng(103)
for code in (4, 3):
    obs = perf_dn[perf_dn.stim == code].correct.mean() - perf_dn[perf_dn.stim == 0].correct.mean()
    boot = gao2018.hierarchical_bootstrap(perf_data(perf_dn, lambda g: np.select([g.stim == 0, g.stim == code], [0, 1], -1)),
                                          2, 10000, rng)
    d = perf_change(boot)
    print(f"DN mice ({perf_dn.subject.nunique()}), code {code}: control {perf_dn[perf_dn.stim == 0].correct.mean():.3f}, "
          f"photostim {perf_dn[perf_dn.stim == code].correct.mean():.3f}; change {obs:+.3f}; 95% interval "
          f"{np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f}; one-tailed p = {gao2018.one_tailed_p(obs, d):.4f}")
print("(code 4 is the test; code 3, sample-epoch dentate photo-activation, is shown for reference)")
pm = perf_dn[perf_dn.stim.isin([0, 4])].groupby(["subject", "stim"]).correct.mean().unstack()
print("Per mouse:"); print((pm[4] - pm[0]).round(3).to_string())

**Result: reproduced.** Delay-epoch dentate photo-activation (0.5 s, three mice) changes
performance by -0.038 (95% interval -0.091 to +0.027, one-tailed p = 0.108), so the test does not
find an effect. The paper's dentate result is from 1.3 s photostimulation in eight mice.

## fig1h-fn-delay-bias-either-direction

> Fastigial photo-activation during the delay epoch biased future movements to either the
> contralateral or the ipsilateral direction in individual mice.

In [ ]:
rng = np.random.default_rng(104)
rows = []
for m, sub in perf_fn.groupby("subject"):
    cond = lambda g: np.select([(g.stim == 0) & (g.side == "contra"), (g.stim == 0) & (g.side == "ipsi"),
                                (g.stim == 2) & (g.side == "contra"), (g.stim == 2) & (g.side == "ipsi")], [0, 1, 2, 3], -1)
    boot = gao2018.hierarchical_bootstrap({m: perf_data(sub, cond)[m]}, 4, 10000, rng)
    bias = (boot[:, 3, 0] / boot[:, 3, 1] - boot[:, 1, 0] / boot[:, 1, 1]) - (boot[:, 2, 0] / boot[:, 2, 1] - boot[:, 0, 0] / boot[:, 0, 1])
    pc = sub[(sub.stim == 0)].groupby("side").correct.mean()
    ps = sub[(sub.stim == 2)].groupby("side").correct.mean()
    b = (ps["ipsi"] - pc["ipsi"]) - (ps["contra"] - pc["contra"])
    p2 = 2 * gao2018.one_tailed_p(b, bias)
    rows.append({"mouse": m, "site": next(s["site_hemisphere"] for s in sessions if s["subject"] == m),
                 "sessions": sub.session.nunique(), "contra change": ps["contra"] - pc["contra"],
                 "ipsi change": ps["ipsi"] - pc["ipsi"], "bias (ipsi - contra)": b, "two_tailed_p": min(p2, 1.0),
                 "n_code2": int((sub.stim == 2).sum())})
bias_tbl = pd.DataFrame(rows)
bias_tbl["direction"] = np.where(bias_tbl.two_tailed_p >= 0.05, "none",
                                 np.where(bias_tbl["bias (ipsi - contra)"] > 0, "ipsilateral", "contralateral"))
print("Bias = change on ipsilateral trials minus change on contralateral trials; positive when contralateral trials lose")
print("more, that is, when photostimulation pushes licking to the ipsilateral side.")
print(bias_tbl.round(4).to_string(index=False))

The direction column names the side toward which photostimulation pushes licking (relative to
the stimulated nucleus), for mice with a two-tailed p below 0.05.

**Result: reproduced.** Of the six fastigial mice, two (BAYLORNL15 and anm00314760) are pushed
toward ipsilateral licking (bias +0.68 and +0.76, p < 0.0001) and one (anm00314746) toward
contralateral licking (bias -0.49, p = 0.0006); the other three show no significant bias (p 0.48
to 0.63).

## ed1f-fn-no-early-licks

> Fastigial photo-activation during the delay epoch did not evoke early licks.

In [ ]:
rng = np.random.default_rng(105)
early = pd.concat([s["trials"].assign(subject=s["subject"], session=s["session"]) for s in sessions if s["group"] == "FN"],
                  ignore_index=True)
early = early[early.stim.isin([0, 2])].assign(correct=lambda t: t.lick_early.astype(float))
obs = early[early.stim == 2].correct.mean() - early[early.stim == 0].correct.mean()
boot = gao2018.hierarchical_bootstrap(perf_data(early, lambda g: np.where(g.stim == 2, 1, 0)), 2, 10000, rng)
d = perf_change(boot)
print(f"Early-lick rate, FN mice: control {early[early.stim == 0].correct.mean():.3f} ({(early.stim == 0).sum()} trials), "
      f"code 2 {early[early.stim == 2].correct.mean():.3f} ({(early.stim == 2).sum()} trials)")
print(f"increase {obs:+.3f}; 95% interval {np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f}; "
      f"one-tailed p for an increase = {float((d <= 0).mean()):.4f}")
pm = early.groupby(["subject", "stim"]).correct.mean().unstack()
print("Per mouse:"); print(pm.round(3).to_string())
print("Variant, no-lick rate (trials without early licks): control",
      round(float((early[(early.stim == 0) & (early.lick_early == 0)].outcome == "nolick").mean()), 3), "code 2",
      round(float((early[(early.stim == 2) & (early.lick_early == 0)].outcome == "nolick").mean()), 3))

**Result: reproduced.** The early-lick rate is 0.079 on control and 0.127 on code 2 trials
(increase +0.049, 95% interval -0.026 to +0.163, one-tailed p = 0.174), so the test does not find
an increase. One mouse (BAYLORCD6) rises from 0.081 to 0.324. The no-lick rate rises from 0.018 to
0.11 on code 2 trials (a variant, not tested).

## fig1k-pi-delay-performance

> CN photo-inhibition during the delay epoch resulted in incorrect behavioural choices.

In [ ]:
rng = np.random.default_rng(106)
for label, codes in (("codes 5 and 6", (5, 6)), ("code 5 (0.5 s)", (5,)), ("code 6 (1.3 s)", (6,))):
    sub = perf_pi[perf_pi.stim.isin((0,) + codes)]
    obs = sub[sub.stim != 0].correct.mean() - sub[sub.stim == 0].correct.mean()
    boot = gao2018.hierarchical_bootstrap(perf_data(sub, lambda g: np.where(g.stim == 0, 0, 1)), 2, 10000, rng)
    d = perf_change(boot)
    print(f"PI mice, {label}: control {sub[sub.stim == 0].correct.mean():.3f}, photo-inhibition {sub[sub.stim != 0].correct.mean():.3f} "
          f"({(sub.stim != 0).sum()} trials); change {obs:+.3f}; 95% interval {np.percentile(d, 2.5):+.3f} to "
          f"{np.percentile(d, 97.5):+.3f}; one-tailed p = {gao2018.one_tailed_p(obs, d):.4f}")
pm = perf_pi.groupby(["subject", "stim"]).correct.mean().unstack()
print("\nPer mouse:"); print(pm.round(3).to_string())

**Result: reproduced.** In the four L7-cre mice, CN photo-inhibition in the delay epoch lowers
performance from 0.848 to 0.693 (change -0.155, 95% interval -0.245 to -0.067, one-tailed p <
0.0001); the 0.5 s (code 5) and 1.3 s (code 6) photostimuli each lower it (-0.142, p = 0.012;
-0.171, p < 0.0001), in every mouse.

## ALM selectivity (Figure 2)

Every pyramidal neuron is tested for selectivity in the sample, delay and response epochs
(two-tailed t-test on spike rates of correct control trials without early licks, in the unit's
stable trials, lick right against lick left).

In [ ]:
sel = []
for s in sessions:
    t = gao2018.selectivity_tests(s, unit_mask=s["cell_type"] == "pyramidal")
    sel.append(t.assign(group=s["group"], subject=s["subject"], session=s["session"]))
sel = pd.concat(sel, ignore_index=True)
for e in ("sample", "delay", "response"):
    sel[f"sig_{e}"] = sel[f"p_{e}"] < gao2018.SELECTIVE_P
sel["selective"] = sel[["sig_sample", "sig_delay", "sig_response"]].any(axis=1)
sel["preparatory"] = sel[["sig_sample", "sig_delay"]].any(axis=1)
untested = sel[["p_sample", "p_delay", "p_response"]].isna().any(axis=1)
print("Pyramidal neurons tested:", len(sel), "; without a test in some epoch (fewer than 2 trials of a type):", int(untested.sum()))
print("Correct control trials per neuron, median (lick right, lick left):", int(sel.n_right.median()), int(sel.n_left.median()))
readings = {"(a) photo-activation mice": sel.group.isin(["FN", "DN"]), "(b) all 13 mice": sel.group.notna()}
frac_rows = []
for r, m in readings.items():
    frac_rows.append({"reading": r, "neurons": int(m.sum()), "selective": sel.selective[m].mean(),
                      "preparatory": sel.preparatory[m].mean(), "peri-movement": sel.sig_response[m].mean()})
fractions = pd.DataFrame(frac_rows).set_index("reading")
print(fractions.round(3).to_string())
m30 = (sel.n_right >= 30) & (sel.n_left >= 30)
for r, m in readings.items():
    print(f"Variant, {r}, neurons with at least 30 correct control trials of each type: {int((m & m30).sum())} neurons, "
          f"selective {sel.selective[m & m30].mean():.3f}, preparatory {sel.preparatory[m & m30].mean():.3f}, "
          f"peri-movement {sel.sig_response[m & m30].mean():.3f}")
print("\nBy mouse (fractions):")
print(sel.groupby(["group", "subject"])[["selective", "preparatory", "sig_response"]].mean().round(3)
      .assign(neurons=sel.groupby(["group", "subject"]).size()).to_string())

## fig2-alm-selective-fraction

> Most ALM pyramidal neurons distinguished lick left and lick right trials (980/1194, p < 0.05,
> two-tailed t-test).

In [ ]:
print("Expected 0.821 +/- 0.082")
for r in readings:
    v = fractions.loc[r, "selective"]
    print(f"{r}: {v:.3f} ({int(round(v * fractions.loc[r, 'neurons']))}/{fractions.loc[r, 'neurons']}); within tolerance: {abs(v - 0.821) <= 0.082}")
# The test at p < 0.05 in three epochs gives a false-positive rate per neuron of up to about 14%;
# with trial labels shuffled within each session the fraction is:
rng = np.random.default_rng(201)
null = []
for s in sessions:
    if s["group"] not in ("FN", "DN"):
        continue
    sh = dict(s, trials=s["trials"].assign(trial_type=rng.permutation(s["trials"].trial_type.values)))
    t = gao2018.selectivity_tests(sh, unit_mask=s["cell_type"] == "pyramidal")
    null.append((t[["p_sample", "p_delay", "p_response"]] < 0.05).any(axis=1))
print(f"Shuffled labels, reading (a): {pd.concat(null).mean():.3f} selective")
print("Threshold variants, reading (a):", {p: round(float((sel[readings['(a) photo-activation mice']][['p_sample', 'p_delay', 'p_response']] < p).any(axis=1).mean()), 3)
                                           for p in (0.01, 0.05)})

**Result: reproduced.** 866 of 1112 pyramidal neurons of the photo-activation mice (0.779) are
selective, within 0.821 +/- 0.082, as are 1365 of 1818 over all 13 mice (0.751). With trial types
shuffled the same test calls 0.121 of neurons selective. With the scripts' rule of at least 30
correct trials of each type the fraction is 0.834 (961 neurons).

## fig2-alm-preparatory-fraction

> Preparatory activity (selectivity before the response epoch) was present in 59% of ALM
> neurons.

In [ ]:
print("Expected 0.59 +/- 0.059")
for r in readings:
    v = fractions.loc[r, "preparatory"]
    print(f"{r}: {v:.3f} ({int(round(v * fractions.loc[r, 'neurons']))}/{fractions.loc[r, 'neurons']}); within tolerance: {abs(v - 0.59) <= 0.059}")
ma = readings["(a) photo-activation mice"]
print("Variant, as a fraction of selective neurons, reading (a):", round(sel.preparatory[ma].sum() / sel.selective[ma].sum(), 3))
print("Delay epoch alone, reading (a):", round(sel.sig_delay[ma].mean(), 3), "; sample alone:", round(sel.sig_sample[ma].mean(), 3))

**Result: reproduced.** Under reading (a), 621 of 1112 pyramidal neurons (0.558) are selective
in the sample or delay epoch, within 0.59 +/- 0.059; over all 13 mice (reading b) the fraction is
0.525, outside it. With at least 30 correct trials of each type, reading (a) gives 0.608.

## fig2-alm-perimovement-fraction

> Peri-movement activity (selectivity in the response epoch) was present in 67% of ALM neurons.

In [ ]:
print("Expected 0.67 +/- 0.067")
for r in readings:
    v = fractions.loc[r, "peri-movement"]
    print(f"{r}: {v:.3f}; within tolerance: {abs(v - 0.67) <= 0.067}")

**Result: reproduced.** Under reading (a), 0.629 of pyramidal neurons are selective in the
response epoch, within 0.67 +/- 0.067; reading (b) gives 0.596, outside it. With at least 30
correct trials of each type, reading (a) gives 0.680.

## fig2c-alm-selectivity-time-course

> Selectivity for lick left and lick right trials emerged gradually in the ALM population
> average, starting during the sample epoch and reaching a maximum in the response epoch.

In [ ]:
rng = np.random.default_rng(202)
tc_centres = np.round(np.arange(-0.4, 3.9 + 1e-9, 0.01), 3)   # from pole in; cue at 2.6 s
curves, dropped = [], 0
for s in sessions:
    if s["group"] not in ("FN", "DN"):
        continue
    tr = s["trials"]
    tested = sel[(sel.session == s["session"]) & (sel.subject == s["subject"])].set_index("unit")
    for i, sp in enumerate(s["spikes"]):
        if i not in tested.index or not tested.loc[i, "selective"]:
            continue
        ok = gao2018.stable_trials(sp, tr) & (tr.lick_early == 0).values & (tr.stim == 0).values & (tr.outcome == "hit").values
        r_idx = np.flatnonzero(ok & (tr.trial_type == "R").values)
        l_idx = np.flatnonzero(ok & (tr.trial_type == "L").values)
        if min(len(r_idx), len(l_idx)) < 15:
            dropped += 1
            continue
        r_idx, l_idx = rng.permutation(r_idx), rng.permutation(l_idx)
        k = gao2018.ALM_SCREEN_TRIALS
        dr = gao2018.epoch_rates(sp, tr)["delay"].values
        pref_r = dr[r_idx[:k]].mean() >= dr[l_idx[:k]].mean()
        rate_r = gao2018.binned_counts(sp, tr.pole_in_time.values[r_idx[k:]], tc_centres, 0.2).mean(0) / 0.2
        rate_l = gao2018.binned_counts(sp, tr.pole_in_time.values[l_idx[k:]], tc_centres, 0.2).mean(0) / 0.2
        curves.append((rate_r - rate_l) if pref_r else (rate_l - rate_r))
curves = np.array(curves)
mean_sel = curves.mean(0)
print(f"Selective neurons used: {len(curves)}; dropped for fewer than 15 correct control trials of a type: {dropped}")
late_sample = (tc_centres >= 1.0) & (tc_centres < 1.3)
boot = np.array([curves[rng.integers(0, len(curves), len(curves))][:, late_sample].mean() for _ in range(2000)])
print(f"(1) mean selectivity, last 300 ms of the sample epoch: {curves[:, late_sample].mean():.2f} spikes/s, "
      f"bootstrap 95% interval {np.percentile(boot, 2.5):.2f} to {np.percentile(boot, 97.5):.2f}")
win = (tc_centres >= 0) & (tc_centres <= 3.9)
t_max = tc_centres[win][np.argmax(mean_sel[win])]
print(f"(2) maximum of the mean selectivity at {t_max:.2f} s from pole in ({mean_sel[win].max():.2f} spikes/s); "
      f"response epoch is 2.6 to 3.9 s: {2.6 <= t_max <= 3.9}")
for lab, (a, b) in {"pre-sample": (-0.4, 0), "sample": (0, 1.3), "delay": (1.3, 2.6), "response": (2.6, 3.9)}.items():
    m = (tc_centres >= a) & (tc_centres < b)
    print(f"  mean selectivity, {lab}: {mean_sel[m].mean():.2f} spikes/s")
# Variant: preferred type from the response epoch of the screening trials.
curves_resp = []
rng_v = np.random.default_rng(203)
for s in sessions:
    if s["group"] not in ("FN", "DN"):
        continue
    tr = s["trials"]
    tested = sel[(sel.session == s["session"]) & (sel.subject == s["subject"])].set_index("unit")
    for i, sp in enumerate(s["spikes"]):
        if i not in tested.index or not tested.loc[i, "selective"]:
            continue
        ok = gao2018.stable_trials(sp, tr) & (tr.lick_early == 0).values & (tr.stim == 0).values & (tr.outcome == "hit").values
        r_idx = rng_v.permutation(np.flatnonzero(ok & (tr.trial_type == "R").values))
        l_idx = rng_v.permutation(np.flatnonzero(ok & (tr.trial_type == "L").values))
        if min(len(r_idx), len(l_idx)) < 15:
            continue
        k = gao2018.ALM_SCREEN_TRIALS
        rr = gao2018.epoch_rates(sp, tr)["response"].values
        pref_r = rr[r_idx[:k]].mean() >= rr[l_idx[:k]].mean()
        rate_r = gao2018.binned_counts(sp, tr.pole_in_time.values[r_idx[k:]], tc_centres, 0.2).mean(0) / 0.2
        rate_l = gao2018.binned_counts(sp, tr.pole_in_time.values[l_idx[k:]], tc_centres, 0.2).mean(0) / 0.2
        curves_resp.append((rate_r - rate_l) if pref_r else (rate_l - rate_r))
mean_resp = np.mean(curves_resp, axis=0)
print(f"Variant, preferred type from the response epoch: maximum at {tc_centres[win][np.argmax(mean_resp[win])]:.2f} s "
      f"({mean_resp[win].max():.2f} spikes/s); late-sample mean {mean_resp[late_sample].mean():.2f}, response-epoch mean "
      f"{mean_resp[(tc_centres >= 2.6) & (tc_centres < 3.9)].mean():.2f} spikes/s")
fig, ax = plt.subplots(figsize=(6, 3))
sem = curves.std(0) / np.sqrt(len(curves))
ax.fill_between(tc_centres, mean_sel - sem, mean_sel + sem, color="0.8"); ax.plot(tc_centres, mean_sel, "k")
for x in (0, 1.3, 2.6):
    ax.axvline(x, color="k", lw=0.5, ls=":")
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("time from pole in (s)"); ax.set_ylabel("selectivity (spikes/s)")
plt.tight_layout(); plt.show()
print("Plotted range:", round(float((mean_sel - sem).min()), 2), "to", round(float((mean_sel + sem).max()), 2), "spikes/s")

**Result: partially reproduced.** Selectivity emerges in the sample epoch (1.19 spikes/s in its
last 300 ms, 95% interval 0.99 to 1.42, against 0.02 before pole in) and grows through the delay
(mean 2.09 spikes/s), but its maximum (2.53 spikes/s) falls at 2.54 s, just before the cue, and
the response-epoch mean (0.96) is lower than the delay mean, so the second part of the criterion
fails. When the preferred type is picked from the response epoch, the maximum is at 3.09 s, in
the response epoch (2.66 spikes/s), and the late-sample selectivity is 0.55 spikes/s; the paper
says the choice of epoch made little difference, which these data do not bear out for the
response epoch. The criterion gave no partial level; the effect holds in part.

## Photostimulation effects on single ALM neurons

For each pyramidal neuron, spike counts on control and photostimulation trials (correct and
error, lick right and lick left pooled, no early licks, stable trials) are compared with a
two-tailed t-test. `unit_trials` gathers what the following claims need.

In [ ]:
def unit_trials(group):
    """Per pyramidal neuron of the group, its usable trials (correct or error, no early lick, stable)."""
    out = []
    for s in sessions:
        if s["group"] != group:
            continue
        tr = s["trials"]
        base = (tr.lick_early == 0).values & tr.outcome.isin(["hit", "error"]).values
        for i, sp in enumerate(s["spikes"]):
            if s["cell_type"][i] != "pyramidal":
                continue
            out.append({"subject": s["subject"], "session": s["session"], "unit": i, "spikes": sp,
                        "trials": tr[base & gao2018.stable_trials(sp, tr)]})
    return out


def modulation(units_, codes, onset_col, window):
    """t-test of spike counts in `window` (s from onset_col) between control and the given codes."""
    rows = []
    for u in units_:
        tr = u["trials"]
        a = tr[onset_col].values + window[0]
        n = gao2018.counts(u["spikes"], a, tr[onset_col].values + window[1]).astype(float)
        c, st = n[tr.stim.values == 0], n[np.isin(tr.stim.values, codes)]
        if len(st) < 2 or len(c) < 2:
            rows.append({"p": np.nan, "diff": np.nan, "ctrl_rate": np.nan, "stim_rate": np.nan, "n_stim": len(st)})
            continue
        w = window[1] - window[0]
        rows.append({"p": stats.ttest_ind(c, st).pvalue, "diff": (st.mean() - c.mean()) / w,
                     "ctrl_rate": c.mean() / w, "stim_rate": st.mean() / w, "n_stim": len(st)})
    return pd.DataFrame(rows)


def change_curves(units_, codes, onset_col, signs, centres, width=0.01):
    """Per neuron, photostimulation minus control rate in bins of `width` around the onset, times `signs`."""
    out = []
    for u, sg in zip(units_, signs):
        tr = u["trials"]
        t0 = tr[onset_col].values
        x = gao2018.binned_counts(u["spikes"], t0, centres, width) / width
        out.append(sg * (x[np.isin(tr.stim.values, codes)].mean(0) - x[tr.stim.values == 0].mean(0)))
    return np.array(out)


ut_pi, ut_fn, ut_dn = unit_trials("PI"), unit_trials("FN"), unit_trials("DN")
lat_centres = np.round(np.arange(-0.015, 0.1 + 1e-9, 0.001), 4)      # bins from 20 ms before onset
pre_centres = np.round(np.arange(-0.2, -0.02 + 1e-9, 0.001), 4)       # for the baseline-subtracted variant
print("Pyramidal neurons: PI", len(ut_pi), "FN", len(ut_fn), "DN", len(ut_dn))

## fig3e-alm-modulated-fraction

> Activity of many ALM neurons was rapidly altered by CN photo-inhibition (197/454, p < 0.01,
> two-tailed t-test).

In [ ]:
pi_mod = {}
for r, codes in (("(a) code 6", (6,)), ("(b) codes 5 and 6", (5, 6))):
    m = modulation(ut_pi, codes, "pole_out_time", (0.0, 1.3))
    pi_mod[r] = m
    sig = m.p < gao2018.PHOTOSTIM_P
    tested = m.p.notna()
    print(f"{r}: {int(sig.sum())}/{int(tested.sum())} neurons modulated = {sig.sum() / tested.sum():.3f} "
          f"(expected 0.434 +/- 0.043; within: {abs(sig.sum() / tested.sum() - 0.434) <= 0.043}); not tested: {int((~tested).sum())}")
m = pi_mod["(a) code 6"]
m["subject"] = [u["subject"] for u in ut_pi]
print("\nBy mouse, reading (a):")
print(m.groupby("subject").apply(lambda g: pd.Series({"neurons": int(g.p.notna().sum()), "modulated": float((g.p < 0.01).mean())}),
                                 include_groups=False).round(3).to_string())
print("Threshold variant, reading (a), p < 0.05:", round(float((m.p < 0.05).sum() / m.p.notna().sum()), 3))
# A control-against-control check of the false-positive rate: control trials split at random in two.
rng = np.random.default_rng(301)
fp = []
for u in ut_pi:
    tr = u["trials"]
    n = gao2018.counts(u["spikes"], tr.pole_out_time.values, tr.cue_time.values)[tr.stim.values == 0]
    k = rng.permutation(len(n))
    fp.append(stats.ttest_ind(n[k[: len(k) // 2]], n[k[len(k) // 2:]]).pvalue < 0.01 if len(n) > 4 else np.nan)
print("Control trials split in two halves, fraction 'modulated':", round(float(np.nanmean(fp)), 3), "(expected 0.01)")

**Result: reproduced.** Under reading (b), codes 5 and 6 pooled, 298 of 695 pyramidal neurons
(0.429) are modulated, within 0.434 +/- 0.043; under reading (a), code 6 alone, 234 of 643 (0.364)
are, outside it. Splitting control trials in two gives 0.01, the false-positive rate of the test.
The fraction differs by mouse (0.18 in BAYLORCD12 and CD13, 0.39 and 0.44 in CD8 and CD14).

## fig3e-alm-mean-rate-unchanged

> CN photo-inhibition left the average ALM spike rate nearly unchanged (control 4.15,
> photo-inhibition 4.39 spikes/s, p = 0.58, two-tailed t-test).

In [ ]:
m = pi_mod["(a) code 6"].dropna(subset=["p"])
print(f"Neurons: {len(m)}; mean delay-epoch rate control {m.ctrl_rate.mean():.2f}, code 6 {m.stim_rate.mean():.2f} spikes/s "
      "(expected 4.15 +/- 0.415 and 4.39 +/- 0.439)")
print(f"Two-sample t-test across neurons: p = {stats.ttest_ind(m.ctrl_rate, m.stim_rate).pvalue:.3f}")
print(f"Variant, paired t-test: p = {stats.ttest_rel(m.ctrl_rate, m.stim_rate).pvalue:.3g}; median change "
      f"{(m.stim_rate - m.ctrl_rate).median():+.3f} spikes/s")
mb = pi_mod["(b) codes 5 and 6"].dropna(subset=["p"])
print(f"Variant, codes 5 and 6: control {mb.ctrl_rate.mean():.2f}, photo-inhibition {mb.stim_rate.mean():.2f} spikes/s, "
      f"two-sample p = {stats.ttest_ind(mb.ctrl_rate, mb.stim_rate).pvalue:.3f}")

**Result: reproduced.** The mean delay-epoch rate is 4.46 spikes/s on control and 4.80 on code
6 trials (both within tolerance of 4.15 and 4.39), and the two-sample t-test gives p = 0.379. The
paired test also finds no change (p = 0.151).

## fig3e-alm-up-down

> Individual ALM neurons increased (108/197) or decreased (89/197) their responses during CN
> photo-inhibition.

In [ ]:
for r, m in pi_mod.items():
    sig = m[m.p < 0.01]
    up = int((sig["diff"] > 0).sum())
    print(f"{r}: {up} increased, {len(sig) - up} decreased; fraction increased {up / len(sig):.3f} "
          f"(expected 0.548 +/- 0.055); binomial p = {stats.binomtest(up, len(sig)).pvalue:.3g}")

**Result: reproduced.** Under reading (b), the one that matched, 172 of 298 modulated neurons
increase (0.577), within 0.548 +/- 0.055; under reading (a), 0.585. More neurons increase than
decrease (binomial p = 0.009), which the paper does not state.

## fig3-alm-latency-pi

> The ALM response to CN photo-inhibition had a latency of 7.1 +/- 3.8 ms (mean +/- s.e.m.).

In [ ]:
rng = np.random.default_rng(302)
lat_results = {}
for r, codes in (("(a) code 6", (6,)), ("(b) codes 5 and 6", (5, 6))):
    m = pi_mod[r]
    sig = (m.p < 0.01).values
    units_sig = [u for u, k in zip(ut_pi, sig) if k]
    ch = change_curves(units_sig, codes, "pole_out_time", np.sign(m["diff"].values[sig]), lat_centres)
    mean, sd, onsets = gao2018.onset_latency(ch, lat_centres, rng)
    lat_results[r] = (mean, sd)
    pre = change_curves(units_sig, codes, "pole_out_time", np.sign(m["diff"].values[sig]), pre_centres).mean(1, keepdims=True)
    mean_b, sd_b, _ = gao2018.onset_latency(ch - pre, lat_centres, rng, n_boot=2000)
    print(f"{r}: {len(units_sig)} neurons; onset {mean * 1000:.1f} +/- {sd * 1000:.1f} ms (bootstrap s.d. over neurons); "
          f"resamples without an onset in the window: {int(np.isnan(onsets).sum())}; before 0 ms: {float((onsets < 0).mean()):.3f}")
    print(f"   variant, pre-onset change subtracted ({float(pre.mean()):.2f} spikes/s): onset {mean_b * 1000:.1f} +/- {sd_b * 1000:.1f} ms")
    print(f"   expected 7.1 +/- 3.8 ms; within: {abs(mean * 1000 - 7.1) <= 3.8}")
    print("   population change (spikes/s) at bin centres -10, 0, 5, 10, 15, 20, 30 ms:",
          [round(float(ch.mean(0)[np.argmin(np.abs(lat_centres - x / 1000))]), 2) for x in (-10, 0, 5, 10, 15, 20, 30)])
# Variants: the first of 10 consecutive significant bins (a sustained change), and the change in 5 ms
# steps from 50 ms before onset, to show where the rise is.
wide_centres = np.round(np.arange(-0.05, 0.1 + 1e-9, 0.005), 3)
for r, codes in (("(a) code 6", (6,)), ("(b) codes 5 and 6", (5, 6)), ("code 5 alone", (5,))):
    m = pi_mod[r] if r in pi_mod else modulation(ut_pi, codes, "pole_out_time", (0.0, 1.3))
    sig = (m.p < 0.01).values
    units_sig = [u for u, k in zip(ut_pi, sig) if k]
    ch = change_curves(units_sig, codes, "pole_out_time", np.sign(m["diff"].values[sig]), lat_centres)
    mean10, sd10, _ = gao2018.onset_latency(ch, lat_centres, rng, n_boot=2000, run=10)
    chw = change_curves(units_sig, codes, "pole_out_time", np.sign(m["diff"].values[sig]), wide_centres).mean(0)
    print(f"{r}: sustained onset {mean10 * 1000:.1f} +/- {sd10 * 1000:.1f} ms; change in 10 ms bins at -50 to +100 ms (5 ms steps):")
    print("   ", np.round(chw, 1).tolist())

**Result: not reproduced.** Under reading (b), the one that matched in
fig3e-alm-modulated-fraction, the onset is 16.6 +/- 12.9 ms, outside 7.1 +/- 3.8 ms. Under reading
(a) it is 7.8 +/- 14.3 ms, within tolerance, but 29% of the resampled onsets fall before the
photostimulus, so this value reflects noise crossings of the threshold before the rise. The
population change stays below 1 spikes/s up to 25 ms after delay onset and rises to 2 to 3
spikes/s between 30 and 40 ms; the sustained onset is 29 to 32 ms for codes 5, 6 and both. The event
times of these sessions lie on a 10 ms grid (data check), which limits any latency to that
precision.

## fig3f-alm-selectivity-abolished

> ALM trial-type selectivity was abolished by CN photo-inhibition.

In [ ]:
def photostim_selectivity(units_, codes, rng, onset_col="pole_out_time", min_stim=4, centres=None):
    """Selectivity (preferred minus non-preferred) on held-out control and on photostimulation trials.

    Neurons selective in the sample or delay epoch (t-test p < 0.05, correct and error trials by
    instructed type, control trials), with at least `min_stim` photostimulation trials of each
    type and 15 control trials of each type; preferred type from the delay rate of 10 random
    control trials of each type.
    """
    rows, curves_c, curves_s = [], [], []
    for u in units_:
        tr, sp = u["trials"], u["spikes"]
        r = gao2018.epoch_rates(sp, tr)
        ctrl, stim = tr.stim.values == 0, np.isin(tr.stim.values, codes)
        R, L = (tr.trial_type == "R").values, (tr.trial_type == "L").values
        if min((ctrl & R).sum(), (ctrl & L).sum()) < 15 or min((stim & R).sum(), (stim & L).sum()) < min_stim:
            continue
        p_s = stats.ttest_ind(r["sample"].values[ctrl & R], r["sample"].values[ctrl & L]).pvalue
        p_d = stats.ttest_ind(r["delay"].values[ctrl & R], r["delay"].values[ctrl & L]).pvalue
        if not (p_s < 0.05 or p_d < 0.05):
            continue
        ri, li = rng.permutation(np.flatnonzero(ctrl & R)), rng.permutation(np.flatnonzero(ctrl & L))
        k = gao2018.ALM_SCREEN_TRIALS
        d = r["delay"].values
        sgn = 1.0 if d[ri[:k]].mean() >= d[li[:k]].mean() else -1.0
        ctrl_sel = sgn * (d[ri[k:]].mean() - d[li[k:]].mean())
        stim_sel = sgn * (d[stim & R].mean() - d[stim & L].mean())
        rows.append({"subject": u["subject"], "ctrl": ctrl_sel, "stim": stim_sel})
        if centres is not None:
            x = gao2018.binned_counts(sp, tr[onset_col].values, centres, 0.2) / 0.2
            curves_c.append(sgn * (x[ri[k:]].mean(0) - x[li[k:]].mean(0)))
            curves_s.append(sgn * (x[stim & R].mean(0) - x[stim & L].mean(0)))
    return pd.DataFrame(rows), np.array(curves_c), np.array(curves_s)


def judge_abolished(tbl, rng, label):
    ratio = tbl.stim.mean() / tbl.ctrl.mean()
    boot = np.array([tbl.iloc[rng.integers(0, len(tbl), len(tbl))].pipe(lambda t: t.stim.mean() - t.ctrl.mean())
                     for _ in range(10000)])
    p = float((boot >= 0).mean())
    print(f"{label}: {len(tbl)} neurons; delay selectivity control {tbl.ctrl.mean():.2f}, photostim {tbl.stim.mean():.2f} "
          f"spikes/s; ratio {ratio:.2f}; one-tailed bootstrap p (photostim not lower) = {p:.4f}; "
          f"95% interval of photostim selectivity {np.percentile([tbl.stim.iloc[rng.integers(0, len(tbl), len(tbl))].mean() for _ in range(2000)], [2.5, 97.5]).round(2).tolist()}")
    return ratio, p


rng = np.random.default_rng(303)
sel_centres = np.round(np.arange(-1.3, 2.6 + 1e-9, 0.02), 3)
pi_sel, pc, ps = photostim_selectivity(ut_pi, (6,), rng, centres=sel_centres)
judge_abolished(pi_sel, rng, "Code 6 (1.3 s)")
print("By mouse:"); print(pi_sel.groupby("subject")[["ctrl", "stim"]].agg(["mean", "size"]).round(2).to_string())
pi_sel_b, _, _ = photostim_selectivity(ut_pi, (5, 6), rng)
judge_abolished(pi_sel_b, rng, "Variant, codes 5 and 6")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(sel_centres, pc.mean(0), "k", label="control"); ax.plot(sel_centres, ps.mean(0), "c", label="CN photo-inhibition (code 6)")
for x in (-1.3, 0, 1.3):
    ax.axvline(x, color="k", lw=0.5, ls=":")
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("time from delay onset (s)"); ax.set_ylabel("selectivity (spikes/s)"); ax.legend()
plt.tight_layout(); plt.show()
print("Plotted range:", round(float(min(pc.mean(0).min(), ps.mean(0).min())), 2), "to",
      round(float(max(pc.mean(0).max(), ps.mean(0).max())), 2), "spikes/s")

**Result: reproduced.** In 252 selective neurons, delay selectivity falls from 2.04 spikes/s on
control trials to 0.48 on code 6 trials, a ratio of 0.23, within the 25% the criterion allows;
the remaining selectivity is above 0 (95% interval 0.21 to 0.76). In two of four mice it falls to
near zero (0.08 and -0.02 spikes/s), in the other two to 0.71 and 0.73. With codes 5 and 6 pooled
the ratio is 0.36.

## ed9a-fn-latency

> Fastigial photo-activation induced rapid changes in ALM activity, with a latency of 4.5 +/- 2.6
> ms, in 227 significantly modulated neurons.

For reading (b) the sample (code 1) and delay (code 2) photostimulation trials are pooled, each
aligned to its own onset: the counts in the 0.5 s after onset are taken as deviations from the
control mean in the same task window, and the change curves are the trial-weighted mean of the
two codes' curves.

In [ ]:
def act_modulation_and_change(units_, codes_onsets, centres):
    """Modulation (0.5 s after onset) and change curves for photo-activation codes, pooled over codes."""
    rows, curves = [], []
    for u in units_:
        tr, sp = u["trials"], u["spikes"]
        ctrl_dev, stim_dev, curve, n_tot = [], [], 0.0, 0
        ok = True
        for code, col in codes_onsets:
            t0 = tr[col].values
            n = gao2018.counts(sp, t0, t0 + 0.5).astype(float)
            c, st = tr.stim.values == 0, tr.stim.values == code
            if st.sum() < 2 or c.sum() < 2:
                ok = False
                break
            mu = n[c].mean()
            ctrl_dev.append(n[c] - mu); stim_dev.append(n[st] - mu)
            x = gao2018.binned_counts(sp, t0, centres, 0.01) / 0.01
            curve = curve + st.sum() * (x[st].mean(0) - x[c].mean(0)); n_tot += st.sum()
        if not ok:
            rows.append({"p": np.nan, "diff": np.nan}); curves.append(np.full(len(centres), np.nan))
            continue
        cd_, sd_ = np.concatenate(ctrl_dev), np.concatenate(stim_dev)
        rows.append({"p": stats.ttest_ind(cd_, sd_).pvalue, "diff": sd_.mean() / 0.5})
        curves.append(curve / n_tot)
    return pd.DataFrame(rows), np.array(curves)


def latency_claim(units_, readings_, rng, label):
    out = {}
    for r, codes_onsets in readings_.items():
        m, curves_ = act_modulation_and_change(units_, codes_onsets, lat_centres)
        sig = (m.p < 0.01).values
        ch = curves_[sig] * np.sign(m["diff"].values[sig])[:, None]
        mean, sd, onsets = gao2018.onset_latency(ch, lat_centres, rng)
        mean10, sd10, _ = gao2018.onset_latency(ch, lat_centres, rng, n_boot=2000, run=10)
        _, pre_curves = act_modulation_and_change([u for u, k in zip(units_, sig) if k], codes_onsets, pre_centres)
        pre = (pre_curves * np.sign(m["diff"].values[sig])[:, None]).mean(1, keepdims=True)
        mean_b, sd_b, _ = gao2018.onset_latency(ch - pre, lat_centres, rng, n_boot=2000)
        early = np.array([u["subject"].startswith("anm") for u in units_])[sig]
        if early.sum() >= 5:
            mean_e, sd_e, _ = gao2018.onset_latency(ch[early], lat_centres, rng, n_boot=2000)
            early_txt = f"2015 files only ({int(early.sum())} neurons), onset {mean_e * 1000:.1f} +/- {sd_e * 1000:.1f} ms"
        else:
            early_txt = f"2015 files only: {int(early.sum())} modulated neurons, too few"
        out[r] = (int(sig.sum()), mean, sd)
        print(f"{label} {r}: {int(sig.sum())} of {int(m.p.notna().sum())} neurons modulated; onset {mean * 1000:.1f} +/- "
              f"{sd * 1000:.1f} ms; resampled onsets before 0 ms: {float((onsets < 0).mean()):.3f}")
        print(f"    variants: sustained onset {mean10 * 1000:.1f} +/- {sd10 * 1000:.1f} ms; with each neuron's change in the 200 to "
              f"20 ms before onset subtracted, onset {mean_b * 1000:.1f} +/- {sd_b * 1000:.1f} ms (pre-onset change "
              f"{float(pre.mean()):.2f} spikes/s); {early_txt}")
        print("    population change (spikes/s) at bin centres -10, 0, 5, 10, 15, 20, 30 ms:",
              [round(float(ch.mean(0)[np.argmin(np.abs(lat_centres - x / 1000))]), 2) for x in (-10, 0, 5, 10, 15, 20, 30)])
    return out

In [ ]:
rng = np.random.default_rng(401)
fn_lat = latency_claim(ut_fn, {"(a) code 2": [(2, "pole_out_time")],
                               "(b) codes 1 and 2": [(1, "pole_in_time"), (2, "pole_out_time")]}, rng, "FN")
for r, (n, mean, sd) in fn_lat.items():
    print(f"{r}: latency within 4.5 +/- 2.6 ms: {abs(mean * 1000 - 4.5) <= 2.6}; count within 227 +/- 22.7: {abs(n - 227) <= 22.7}")

**Result: partially reproduced.** Under reading (a), delay photostimulation, the onset is 2.0
+/- 8.1 ms, within 4.5 +/- 2.6 ms, but 187 neurons are modulated, outside 227 +/- 22.7; under
reading (b) 248 neurons are modulated, within tolerance, and the onset is -0.9 +/- 11.9 ms,
outside it. The first-crossing onset is fragile here: under reading (a), 24% of resampled onsets
fall before the photostimulus. The population change rises from -0.14 spikes/s at 0 ms to 1.65 at
5 ms and 4.6 at 10 ms; the sustained onset is 6.7 +/- 2.3 ms, and in the 2015 files, whose event
times are precise, the first-crossing onset is 9.5 +/- 6.0 ms.

## ed9a-dn-latency

> Dentate photo-activation also induced ALM response changes, with a latency of 8.3 +/- 5.0 ms,
> in 163 significantly modulated neurons.

In [ ]:
rng = np.random.default_rng(402)
dn_lat = latency_claim(ut_dn, {"(a) code 4": [(4, "pole_out_time")],
                               "(b) codes 3 and 4": [(3, "pole_in_time"), (4, "pole_out_time")]}, rng, "DN")
for r, (n, mean, sd) in dn_lat.items():
    print(f"{r}: latency within 8.3 +/- 5.0 ms: {abs(mean * 1000 - 8.3) <= 5.0}; count within 163 +/- 16.3: {abs(n - 163) <= 16.3}")

**Result: not reproduced.** Neither reading gives a latency or count within tolerance: the
onsets are -9.0 +/- 5.7 ms (code 4) and -12.2 +/- 4.5 ms (codes 3 and 4), against 8.3 +/- 5.0,
and 183 and 230 neurons are modulated, against 163 +/- 16.3. In over 96% of resamples the onset
falls before the photostimulus: the population change is already 0.79 spikes/s 10 ms before onset
and 1.31 at onset, and subtracting each neuron's change in the 200 ms before onset still gives
-5.0 ms. Eleven of the 13 dentate sessions are from 2017 files with event times on a 10 ms grid,
and the 2015 files have too few modulated neurons (1 and 4) to measure an onset, so the data
cannot place this onset to within the few milliseconds the claim needs.

## fig4f-fn-persistent

> Transient photo-activation of the fastigial nucleus altered ALM preparatory activity long (more
> than 1 s) after the photostimulus ended.

In [ ]:
fr_centres = np.round(np.arange(-1.1, 2.6 + 1e-9, 0.05), 3)   # 400 ms bins, centres from delay onset


def fraction_modulated(units_, code):
    sig = []
    for u in units_:
        tr = u["trials"]
        x = gao2018.binned_counts(u["spikes"], tr.pole_out_time.values, fr_centres, 0.4)
        c, st = x[tr.stim.values == 0], x[tr.stim.values == code]
        if len(st) < 2:
            continue
        with np.errstate(all="ignore"):
            p = stats.ttest_ind(c, st, axis=0).pvalue
        sig.append(np.nan_to_num(p, nan=1.0) < 0.01)
    return np.array(sig)


sig_fn, sig_dn = fraction_modulated(ut_fn, 2), fraction_modulated(ut_dn, 4)
pre_bin = np.argmin(np.abs(fr_centres - (-0.2)))    # 400 ms ending at photostimulus onset
post_bin = np.argmin(np.abs(fr_centres - 1.7))      # 1.0 to 1.4 s after offset (offset at 0.5 s)
frac_tbl = pd.DataFrame({"centre_s": fr_centres, "FN": sig_fn.mean(0), "DN": sig_dn.mean(0)})
print("Neurons: FN", len(sig_fn), "DN", len(sig_dn))
print("Fraction modulated (p < 0.01) per 400 ms bin, every fourth bin; centres from delay onset (cue at 1.3 s):")
print(frac_tbl.iloc[::4].round(3).to_string(index=False))

In [ ]:
a, b = int(sig_fn[:, post_bin].sum()), int(sig_fn[:, pre_bin].sum())
n = len(sig_fn)
p = stats.fisher_exact([[a, n - a], [b, n - b]], alternative="greater").pvalue
print(f"FN, bin ending at photostimulus onset ({fr_centres[pre_bin] - 0.2:.1f} to {fr_centres[pre_bin] + 0.2:.1f} s): {b}/{n} = {b / n:.3f}")
print(f"FN, bin 1.0 to 1.4 s after offset ({fr_centres[post_bin] - 0.2:.1f} to {fr_centres[post_bin] + 0.2:.1f} s): {a}/{n} = {a / n:.3f}")
print(f"ratio {a / max(b, 1):.1f}; one-tailed Fisher exact p = {p:.2g}")
for lag_s in (0.5, 0.8, 1.2):
    k = np.argmin(np.abs(fr_centres - (0.5 + lag_s + 0.2)))
    print(f"  variant, bin starting {lag_s} s after offset: {sig_fn[:, k].mean():.3f}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(fr_centres, frac_tbl.FN, "b", label="fastigial (code 2)"); ax.plot(fr_centres, frac_tbl.DN, "r", label="dentate (code 4)")
ax.axvspan(0, 0.5, color="c", alpha=0.3); ax.axvline(1.3, color="k", lw=0.5, ls=":")
ax.set_xlabel("400 ms bin centre from delay onset (s)"); ax.set_ylabel("fraction modulated"); ax.legend()
plt.tight_layout(); plt.show()
print("Plotted range:", round(float(frac_tbl[["FN", "DN"]].min().min()), 3), "to", round(float(frac_tbl[["FN", "DN"]].max().max()), 3))

**Result: reproduced.** In the bin 1.0 to 1.4 s after the fastigial photostimulus ended, 78 of
538 neurons (0.145) differ from control, against 5 of 538 (0.009) in the bin ending at onset (one-
tailed Fisher p = 2.3e-19). The fraction declines from 0.361 during photostimulation to 0.119 at
1.2 s after offset.

## fig4g-dn-recovery

> ALM activity quickly recovered after dentate photo-activation.

In [ ]:
a_d, b_d = int(sig_dn[:, post_bin].sum()), int(sig_dn[:, pre_bin].sum())
n_d = len(sig_dn)
p_vs_fn = stats.fisher_exact([[a_d, n_d - a_d], [a, n - a]]).pvalue
print(f"DN, bin ending at onset: {b_d}/{n_d} = {b_d / n_d:.3f}; bin 1.0 to 1.4 s after offset: {a_d}/{n_d} = {a_d / n_d:.3f} "
      f"(ratio {a_d / max(b_d, 1):.1f})")
print(f"FN in the same bin: {a / n:.3f}; two-tailed Fisher exact p (DN against FN) = {p_vs_fn:.2g}")
during = np.argmin(np.abs(fr_centres - 0.25))
print(f"During photostimulation (bin centred 0.25 s): FN {sig_fn[:, during].mean():.3f}, DN {sig_dn[:, during].mean():.3f}")
print("By mouse, DN, post-offset bin:", {m: round(float(sig_dn[[u['subject'] == m for u in ut_dn]][:, post_bin].mean()), 3)
                                         for m in sorted({u['subject'] for u in ut_dn})})

**Result: partially reproduced.** In the bin 1.0 to 1.4 s after offset, 0.051 of dentate-mouse
neurons differ from control, lower than the fastigial 0.145 (Fisher p = 2e-07) and down from
0.328 during photostimulation, but 2.3 times the 0.022 before onset, above the factor of 2 the
criterion allows. ALM recovers faster after dentate than after fastigial photo-activation, and
not fully within 1.4 s. The criterion gave no partial level; the effect holds in part.

## Coding direction (Figure 4i, Extended Data Figure 10d)

`cd_sessions` computes, for each session, the coding direction from a random half of the correct
control trials and the projections of the other control trials and the photostimulation trials.
All pyramidal neurons of a session and all its usable trials (correct or error, no early lick)
are used; the stable-trial rule is not applied here, since requiring every trial to be stable
for every neuron leaves too few trials (a choice made here).

In [ ]:
cd_centres = np.round(np.arange(0.2, 1.1 + 1e-9, 0.01), 3)    # 400 ms windows inside the delay, from delay onset
traj_centres = np.round(np.arange(-1.1, 1.3 + 1e-9, 0.05), 3)


def cd_sessions(group, code, rng):
    out = []
    for s in sessions:
        if s["group"] != group:
            continue
        tr = s["trials"]
        base = (tr.lick_early == 0).values & tr.outcome.isin(["hit", "error"]).values
        keep_u = [i for i in range(len(s["spikes"])) if s["cell_type"][i] == "pyramidal"]
        t = tr[base]
        ctrl_hit = np.flatnonzero((t.stim.values == 0) & (t.outcome.values == "hit"))
        R, L = t.trial_type.values == "R", t.trial_type.values == "L"
        n_ok = min(((t.stim.values == code) & R).sum(), ((t.stim.values == code) & L).sum(),
                   ((t.stim.values == 0) & R).sum(), ((t.stim.values == 0) & L).sum())
        if len(keep_u) <= 5 or n_ok <= 5:
            continue
        train = np.zeros(len(t), bool)
        for side in (R, L):
            idx = rng.permutation(np.intersect1d(ctrl_hit, np.flatnonzero(side)))
            train[idx[: len(idx) // 2]] = True
        X = np.stack([gao2018.binned_counts(s["spikes"][i], t.pole_out_time.values, cd_centres, 0.4) for i in keep_u], axis=2)
        cd = (X[train & R].mean(0) - X[train & L].mean(0)).mean(0)
        last = np.stack([gao2018.binned_counts(s["spikes"][i], t.cue_time.values, [-0.2], 0.4)[:, 0] for i in keep_u], axis=1)
        traj = np.stack([gao2018.binned_counts(s["spikes"][i], t.pole_out_time.values, traj_centres, 0.4) for i in keep_u], axis=2)
        proj_last, proj_traj = last @ cd, traj @ cd
        offset = proj_traj[train].mean()
        ep_r, ep_l = proj_last[train & R].mean(), proj_last[train & L].mean()
        test_ctrl, stim = (~train) & (t.stim.values == 0), t.stim.values == code
        licked_r = ((t.outcome.values == "hit") & R) | ((t.outcome.values == "error") & L)
        out.append({"subject": s["subject"], "session": s["session"], "neurons": len(keep_u),
                    "sep_ctrl": proj_last[test_ctrl & R].mean() - proj_last[test_ctrl & L].mean(),
                    "sep_stim": proj_last[stim & R].mean() - proj_last[stim & L].mean(),
                    "traj": {k: (proj_traj[m & R].mean(0) - offset, proj_traj[m & L].mean(0) - offset)
                             for k, m in (("ctrl", test_ctrl), ("stim", stim))},
                    "x_ctrl": (proj_last[test_ctrl] - (ep_r + ep_l) / 2) / (ep_r - ep_l), "y_ctrl": licked_r[test_ctrl],
                    "x_stim": (proj_last[stim] - (ep_r + ep_l) / 2) / (ep_r - ep_l), "y_stim": licked_r[stim],
                    "n_stim": int(stim.sum()), "n_test_ctrl": int(test_ctrl.sum())})
    return pd.DataFrame(out)


rng = np.random.default_rng(501)
cd_fn, cd_dn = cd_sessions("FN", 2, rng), cd_sessions("DN", 4, rng)
n_fn_sessions = len({(s["subject"], s["session"]) for s in sessions if s["group"] == "FN"})
n_dn_sessions = len({(s["subject"], s["session"]) for s in sessions if s["group"] == "DN"})
print(f"Sessions used: FN {len(cd_fn)} of {n_fn_sessions}, DN {len(cd_dn)} of {n_dn_sessions} "
      "(paper, 28 of 39 sessions over both groups and the excluded mouse)")
print("Neurons per session: FN", cd_fn.neurons.tolist(), "; DN", cd_dn.neurons.tolist())
print(cd_fn[["subject", "session", "neurons", "sep_ctrl", "sep_stim", "n_stim"]].round(2).to_string(index=False))
print(cd_dn[["subject", "session", "neurons", "sep_ctrl", "sep_stim", "n_stim"]].round(2).to_string(index=False))


def sep_ratio(tbl, idx):
    t = tbl.iloc[idx]
    return t.sep_stim.mean() / t.sep_ctrl.mean()


rng = np.random.default_rng(502)
boot_fn = np.array([sep_ratio(cd_fn, rng.integers(0, len(cd_fn), len(cd_fn))) for _ in range(10000)])
boot_dn = np.array([sep_ratio(cd_dn, rng.integers(0, len(cd_dn), len(cd_dn))) for _ in range(10000)])
ratio_fn, ratio_dn = sep_ratio(cd_fn, np.arange(len(cd_fn))), sep_ratio(cd_dn, np.arange(len(cd_dn)))

## fig4i-fn-cd-collapse

> Fastigial photo-activation resulted in a collapse of ALM activity trajectories along the coding
> direction, destroying the coding of future movement.

In [ ]:
print(f"FN: separation in the last 400 ms of the delay, control {cd_fn.sep_ctrl.mean():.2f}, code 2 {cd_fn.sep_stim.mean():.2f} "
      f"(spike-count units); ratio {ratio_fn:.3f}, bootstrap 95% interval {np.percentile(boot_fn, 2.5):.3f} to {np.percentile(boot_fn, 97.5):.3f}")
print(f"Variant, the median over sessions of each session's ratio: {np.median(cd_fn.sep_stim / cd_fn.sep_ctrl):.3f}; "
      f"each session's separation scaled by its control separation, mean ratio {np.mean(cd_fn.sep_stim / cd_fn.sep_ctrl):.3f}")
print("Sessions with a smaller separation on code 2 trials:", int((cd_fn.sep_stim < cd_fn.sep_ctrl).sum()), "of", len(cd_fn))
print("By mouse, ratio:", cd_fn.groupby("subject").apply(lambda t: round(t.sep_stim.mean() / t.sep_ctrl.mean(), 2), include_groups=False).to_dict())
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
rng_lo, rng_hi = np.inf, -np.inf
for ax, (lab, tbl) in zip(axes, (("fastigial (code 2)", cd_fn), ("dentate (code 4)", cd_dn))):
    for cond, ls in (("ctrl", "-"), ("stim", "--")):
        r_ = np.mean([t[cond][0] for t in tbl.traj], axis=0); l_ = np.mean([t[cond][1] for t in tbl.traj], axis=0)
        ax.plot(traj_centres, r_, "b" + ls, label=f"lick right, {cond}"); ax.plot(traj_centres, l_, "r" + ls, label=f"lick left, {cond}")
        rng_lo, rng_hi = min(rng_lo, r_.min(), l_.min()), max(rng_hi, r_.max(), l_.max())
    ax.axvspan(0, 0.5, color="c", alpha=0.2); ax.set_title(lab); ax.set_xlabel("400 ms window centre from delay onset (s)")
axes[0].set_ylabel("projection on cd (offset removed)"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Plotted range:", round(float(rng_lo), 2), "to", round(float(rng_hi), 2))

**Result: reproduced.** Over 15 sessions, the separation of lick right and lick left trajectories
in the last 400 ms of the delay is 62.97 on control and 31.42 on code 2 trials, a ratio of 0.499
(95% interval 0.204 to 0.955), just under the 0.5 of the criterion. The result is near that
threshold: the median of the per-session ratios is 0.602, the mean of them 0.854, and two of six
mice (BAYLORCD6, 2.45; anm00314746, 1.27) show no collapse. Separation is smaller with
photostimulation in 11 of 15 sessions. With the dentate sessions the analysis uses 28 sessions,
the number the Methods give.

## fig4i-dn-cd-minimal

> Dentate photo-activation minimally impacted ALM activity trajectories along the coding
> direction, despite having an impact on ALM neuron responses.

In [ ]:
print(f"DN: separation control {cd_dn.sep_ctrl.mean():.2f}, code 4 {cd_dn.sep_stim.mean():.2f}; ratio {ratio_dn:.2f}, "
      f"bootstrap 95% interval {np.percentile(boot_dn, 2.5):.2f} to {np.percentile(boot_dn, 97.5):.2f}")
print(f"Fraction of paired resamples with the FN ratio below the DN ratio: {float((boot_fn < boot_dn).mean()):.4f}")
print(f"Variant, the median over sessions of each session's ratio: {np.median(cd_dn.sep_stim / cd_dn.sep_ctrl):.3f}")
print("By mouse, ratio:", cd_dn.groupby("subject").apply(lambda t: round(t.sep_stim.mean() / t.sep_ctrl.mean(), 2), include_groups=False).to_dict())
print("Sessions with a smaller separation on code 4 trials:", int((cd_dn.sep_stim < cd_dn.sep_ctrl).sum()), "of", len(cd_dn))

**Result: reproduced.** Over 13 sessions the separation is 55.79 on control and 58.21 on code 4
trials, a ratio of 1.04 (95% interval 0.87 to 1.20), and the fastigial ratio is lower in 98.5% of
paired resamples.

## ed10d-fn-no-prediction

> After fastigial photo-activation, ALM activity along the coding direction no longer predicted
> the future movement direction (n = 337 trials).

In [ ]:
import statsmodels.api as sm


def logit(x, y):
    fit = sm.Logit(y.astype(float), sm.add_constant(x)).fit(disp=0)
    return fit.params[1], fit.pvalues[1], len(y)


for lab, tbl in (("FN", cd_fn), ("DN", cd_dn)):
    xc, yc = np.concatenate(tbl.x_ctrl.values), np.concatenate(tbl.y_ctrl.values)
    xs, ys = np.concatenate(tbl.x_stim.values), np.concatenate(tbl.y_stim.values)
    bc, pc_, nc = logit(xc, yc)
    bs, ps_, ns = logit(xs, ys)
    print(f"{lab}: held-out control trials, slope {bc:.2f} (p = {pc_:.2g}, {nc} trials); photostimulation trials, slope {bs:.2f} "
          f"(p = {ps_:.2g}, {ns} trials)")
    bins = np.array([-np.inf, -0.5, 0, 0.5, np.inf])
    print("   P(lick right) by projection bin (<-0.5, -0.5..0, 0..0.5, >0.5): control",
          [round(float(yc[(xc >= a) & (xc < b)].mean()), 2) if ((xc >= a) & (xc < b)).sum() else None for a, b in zip(bins[:-1], bins[1:])],
          "photostim", [round(float(ys[(xs >= a) & (xs < b)].mean()), 2) if ((xs >= a) & (xs < b)).sum() else None for a, b in zip(bins[:-1], bins[1:])])
print("Expected FN trial count 337 +/- 33.7")

**Result: not reproduced.** On fastigial photostimulation trials the projection on the coding
direction still predicts licking right (slope 0.40, p = 0.00015, 386 trials), though less than on
control trials (slope 1.44, p = 3e-63). The probability of licking right goes from 0.51 to 0.77
across projection bins on photostimulation trials, against 0.27 to 0.85 on control trials. The
trial count (386) is outside 337 +/- 33.7.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| stim-code-link | reproduced |
| methods-alm-unit-counts | not reproduced |
| fig1g-fn-delay-performance | reproduced |
| fig1-fn-sample-no-effect | reproduced |
| fig1g-dn-little-effect | reproduced |
| fig1h-fn-delay-bias-either-direction | reproduced |
| ed1f-fn-no-early-licks | reproduced |
| fig1k-pi-delay-performance | reproduced |
| fig2-alm-selective-fraction | reproduced |
| fig2-alm-preparatory-fraction | reproduced |
| fig2-alm-perimovement-fraction | reproduced |
| fig2c-alm-selectivity-time-course | partially reproduced |
| fig3e-alm-modulated-fraction | reproduced |
| fig3e-alm-mean-rate-unchanged | reproduced |
| fig3e-alm-up-down | reproduced |
| fig3-alm-latency-pi | not reproduced |
| fig3f-alm-selectivity-abolished | reproduced |
| ed9a-fn-latency | partially reproduced |
| ed9a-dn-latency | not reproduced |
| fig4f-fn-persistent | reproduced |
| fig4g-dn-recovery | partially reproduced |
| fig4i-fn-cd-collapse | reproduced |
| fig4i-dn-cd-minimal | reproduced |
| ed10d-fn-no-prediction | not reproduced |
| fig1d-fn-lesion, fig1e-lesion-lick-timing, fig1h-response-contra-bias, fig1i-licking-unaffected, fig1k-response-ipsi-bias, ed2-pi-silences-fastigial, ed10c-alm-pi-endpoints, ed10efg-alm-needed-after-fn, fig4bc-fastigial-thalamus | blocked, data missing |

The CN claims are summarised in the second notebook.

**How the analysis differs from the paper's.** Behaviour is tested on the trials of the recording
sessions only, the only behaviour in the dandiset: delay dentate photo-activation is 0.5 s in 3 mice
here against 1.3 s in 8 in Figure 1g, and CN photo-inhibition covers 4 of the 7 mice of Figure 1k.
The dandiset holds more units than the paper counts and does not say which the paper used, so all
units labelled pyramidal are used. The trial table has no column for periods when the mouse was not
performing. The scripts' stable-trial rule, written for the CN data, is applied to ALM, except in
the coding-direction analysis. Ambiguous numbers are tested under the readings listed in the claims
file: reading (b) matched for the photo-inhibition fraction, and the latency and up-down claims are
judged under it. Latencies use a search window and bin centre chosen here.

**What was found about the dandiset.** The photostimulation codes have no description; their names
come from the Zenodo data description and match the ALM responses. The trial time columns are
described as relative to the trial start and hold session times. In three of four photo-inhibition
mice the stimulated or recorded hemisphere disagrees with Supplemental Table 1. The dandiset holds
more units than the paper reports, with no record of which were analysed. In the 40 sessions of the
2017 and 2018 mice the event times lie on a 10 ms grid, too coarse for the paper's millisecond
latencies. The CN recordings, lick times, lesion and behaviour-only sessions and anatomy are not in
the dandiset; the CN recordings are on Zenodo.